# Gold para Power BI — Olist E-commerce

### Importando bibliotecas

In [0]:
import pandas as pd
import numpy as np

In [0]:
customers = spark.table("olist.silver.customers")
geolocation = spark.table("olist.silver.geolocation")
order_items = spark.table("olist.silver.order_items")
order_payments = spark.table("olist.silver.order_payments")
order_reviews = spark.table("olist.silver.order_reviews")
orders = spark.table("olist.silver.orders")
products = spark.table("olist.silver.products")
sellers = spark.table("olist.silver.sellers")
# convertendo para pandas
customers = customers.toPandas()
geolocation = geolocation.toPandas()
order_items = order_items.toPandas()
order_payments = order_payments.toPandas()
order_reviews = order_reviews.toPandas()
orders = orders.toPandas()
products = products.toPandas()
sellers = sellers.toPandas()



### Verificacao das tabelas silver

Verificamos `nunique()`, `value_counts()` e `isnull().sum()` em todas as 8 tabelas silver. As inconsistencias encontradas foram:

| Tabela | Problema | Tratamento |
| --- | --- | --- |
| geolocation | 981 mil duplicatas de CEP | agrupar por CEP (media lat/lng) |
| order_payments | 3 linhas com payment_type = not_defined | remover |
| order_reviews | 88% nulos no titulo, 59% na mensagem | preencher com 'Sem comentario' |
| orders | nulos em datas de entrega (pedidos nao entregues) | manter (esperado) |
| products | 610 sem categoria, 2 sem dimensoes | preencher com 'sem_categoria' e mediana |
| customers | sem nulos | sem tratamento |
| order_items | sem nulos | sem tratamento |
| sellers | sem nulos | sem tratamento |

Os tratamentos serao aplicados na secao a seguir.

### Tratamentos

In [0]:
# geolocation: deduplicar por CEP (1M linhas → ~19K)
geolocation = geolocation.groupby("geolocation_zip_code_prefix").agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean",
    "geolocation_city": "first",
    "geolocation_state": "first",
}).reset_index()

In [0]:
# order_payments: remover 3 linhas com payment_type = 'not_defined'
order_payments = order_payments[order_payments["payment_type"] != "not_defined"]

In [0]:
# order_reviews: preencher nulos nos comentários
order_reviews["review_comment_title"] = order_reviews["review_comment_title"].fillna("Sem comentario")
order_reviews["review_comment_message"] = order_reviews["review_comment_message"].fillna("Sem comentario")

In [0]:
# products: preencher categoria nula e dimensoes nulas
products["product_category_name"] = products["product_category_name"].fillna("sem_categoria")
products["product_category_name_english"] = products["product_category_name_english"].fillna("no_category")
products["product_name_length"] = products["product_name_length"].fillna(products["product_name_length"].median())
products["product_description_length"] = products["product_description_length"].fillna(products["product_description_length"].median())
products["product_photos_qty"] = products["product_photos_qty"].fillna(products["product_photos_qty"].median())
products["product_weight_g"] = products["product_weight_g"].fillna(products["product_weight_g"].median())
products["product_length_cm"] = products["product_length_cm"].fillna(products["product_length_cm"].median())
products["product_height_cm"] = products["product_height_cm"].fillna(products["product_height_cm"].median())
products["product_width_cm"] = products["product_width_cm"].fillna(products["product_width_cm"].median())

### Escrevendo no gold

In [0]:
# convertendo de pandas para spark e salvando no gold
tabelas = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
}

for nome, df in tabelas.items():
    spark.createDataFrame(df).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"olist.gold.powerbi_{nome}")
    print(f"✓ {nome}: {len(df):,} linhas")

✓ customers: 99,441 linhas
✓ geolocation: 19,015 linhas
✓ order_items: 112,650 linhas
✓ order_payments: 103,883 linhas
✓ order_reviews: 99,224 linhas
✓ orders: 99,441 linhas
✓ products: 32,951 linhas
✓ sellers: 3,095 linhas


In [0]:
%sql
SHOW TABLES IN olist.gold

database,tableName,isTemporary
gold,customers,false
gold,geolocation,false
gold,order_items,false
gold,order_payments,false
gold,order_reviews,false
gold,orders,false
gold,powerbi_customers,false
gold,powerbi_geolocation,false
gold,powerbi_order_items,false
gold,powerbi_order_payments,false
